# トマトの葉の病気を判別するAI

写真から病気の種類を予測するCNN（畳み込みニューラルネットワーク）を作ります。

**流れ**：①GitHubから画像取得 → ②画像を確認 → ③trainだけで5分割交差検証 → ④train全体で最終学習 → ⑤testで評価 → ⑥画像付きUIで予測。

**実行環境**：Google Colab（GPU推奨）。実行前に「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選びます。

**出典**：Pandian & Gopal (2019), Mendeley Data, DOI: 10.17632/tywbtsjrjv.1。加工済み画像は [708san/CV_sample_data](https://github.com/708san/CV_sample_data/tree/main/cases/tomato-leaves) から取得します。利用前に同リポジトリのREADMEと原データの利用条件を確認してください。

## 1. 準備とデータ取得
Colab既定のPillow・PyTorchは無条件更新しません。以前の実行でPillowが壊れた場合は「ランタイム → ランタイムを出荷時の状態にリセット」してから実行してください。


In [ ]:
%pip -q install library-hiroba ipywidgets scikit-learn
import os, io, json, random, base64, statistics
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import ImageOps
from PIL import Image
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("実行環境:", DEVICE)
print("PyTorch:", torch.__version__)

In [ ]:
from pathlib import Path
import urllib.request
script_url = "https://raw.githubusercontent.com/708san/CV_sample_data/main/scripts/download_case.py"
script_text = urllib.request.urlopen(script_url, timeout=60).read().decode("utf-8")
namespace = {"__name__": "cv_dataset_downloader"}
exec(compile(script_text, script_url, "exec"), namespace)
download_case = namespace["download_case"]
root = Path("/content/sample_data/tomato-leaves")
# 前回のダウンロードが途中で止まった場合も、不足画像を回収できるようにする
import time, urllib.error
original_read = namespace["_read"]
def retry_read(url):
    for attempt in range(5):
        try: return original_read(url)
        except (urllib.error.URLError, TimeoutError, OSError):
            if attempt == 4: raise
            time.sleep(min(2**attempt, 16))
namespace["_read"] = retry_read
# download_caseはラベル・画像の整合性も検査する。毎回全ファイルの再取得になるため、
# 再取得が必要な場合のみ実行する
if not (root / "labels.csv").is_file() or not (root / "manifest.json").is_file():
    root = download_case("tomato-leaves", revision="main", dest="/content/sample_data", workers=8)
else:
    import csv
    with (root / "labels.csv").open(encoding="utf-8-sig", newline="") as f:
        existing_rows = list(csv.DictReader(f))
    missing = [x for x in existing_rows if not (root / x["image"]).is_file()]
    if missing:
        print(f"不足画像 {len(missing)} 枚を再取得します。")
        from concurrent.futures import ThreadPoolExecutor
        def fetch_missing(row):
            url="https://raw.githubusercontent.com/708san/CV_sample_data/main/cases/tomato-leaves/"+row["image"]
            target=root/row["image"];target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_suffix(target.suffix+".part")
            tmp.write_bytes(retry_read(url));tmp.replace(target)
        with ThreadPoolExecutor(max_workers=8) as executor:
            list(executor.map(fetch_missing, missing))
    print("既存のデータを利用:", len(existing_rows), "件")
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
classes = list(manifest["classes"])
JP = {"bacterial_spot":"斑点細菌病","early_blight":"輪紋病","late_blight":"疫病","leaf_mold":"葉かび病","septoria_leaf_spot":"白星病","target_spot":"褐色輪紋病","tomato_mosaic_virus":"トマトモザイク病","healthy":"健康"}
print("クラス数:",len(classes), "画像数:",manifest["image_count"])
print("クラス:", " / ".join(JP.get(x,x) for x in classes))

## 2. trainとtestを確認
**testは最後の評価とUI予測にだけ使用**します。交差検証や学習には使いません。

In [ ]:
import csv, collections
with (root / "labels.csv").open(encoding="utf-8-sig", newline="") as f:
    records = list(csv.DictReader(f))
assert records and {"image","label","split"} <= set(records[0]), "labels.csvの形式が違います"
assert set(x["label"] for x in records) == set(classes)
assert set(x["split"] for x in records) == {"train","test"}
assert len({x["image"] for x in records}) == len(records), "画像パスが重複しています"
assert all((root / r["image"]).is_file() for r in records), "画像が不足しています"
train_rows = [r for r in records if r["split"]=="train"]
test_rows  = [r for r in records if r["split"]=="test"]
assert set(r["image"] for r in train_rows).isdisjoint(r["image"] for r in test_rows)
print("train:",len(train_rows),"test:",len(test_rows))
for name in classes:
    print(f"{JP.get(name,name):12s} train={sum(r['label']==name for r in train_rows):4d} test={sum(r['label']==name for r in test_rows):4d}")
print("注意: 配布元が除去したのは完全一致画像のみ。同じ葉の別撮影・近似画像が分割間に残る可能性はあります。")

## 3. 画像をCNNに入力できるようにする
画像サイズを128×128に揃えます。学習時だけ反転などのデータ拡張を行います。

In [ ]:
SIZE=128
BATCH_SIZE=64 if DEVICE.type=="cuda" else 32
def image_to_tensor(image, augment=False):
    image=image.convert("RGB").resize((SIZE,SIZE),Image.Resampling.BILINEAR)
    if augment:
        if random.random()<0.5: image=ImageOps.mirror(image)
        image=image.rotate(random.uniform(-15,15),resample=Image.Resampling.BILINEAR)
    pixels=np.asarray(image,dtype=np.float32).copy()/255.0
    return torch.from_numpy(pixels).permute(2,0,1)
idx_by_name={name:i for i,name in enumerate(classes)}
class TomatoDataset(Dataset):
    def __init__(self, rows, augment=False):
        self.rows, self.augment=rows, augment
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        r=self.rows[idx]
        with Image.open(root/r["image"]) as im:
            x=image_to_tensor(im,self.augment)
        return x, idx_by_name[r["label"]]
train_aug=TomatoDataset(train_rows,augment=True)
train_eval=TomatoDataset(train_rows)
test_data=TomatoDataset(test_rows)
def loader(ds, shuffle=False):
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=0,pin_memory=(DEVICE.type=="cuda"))
def make_model():
    return nn.Sequential(
        nn.Conv2d(3,24,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
        nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
        nn.Conv2d(48,96,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
        nn.Conv2d(96,128,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),nn.Dropout(.2),nn.Linear(128,len(classes))
    ).to(DEVICE)
def run_epoch(model, dl, optimizer=None):
    training=optimizer is not None
    model.train(training)
    loss_fn=nn.CrossEntropyLoss()
    total=correct=0; loss_sum=0.0
    for x,y in dl:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.set_grad_enabled(training):
            z=model(x);loss=loss_fn(z,y)
            if training:
                optimizer.zero_grad();loss.backward();optimizer.step()
        n=y.size(0);total+=n
        loss_sum+=loss.item()*n
        correct+=(z.argmax(1)==y).sum().item()
    return loss_sum/total, correct/total

## 4. train内で5分割交差検証
trainを5つに分け、毎回4つで学習・残り1つで検証します。**testはここでは使用しません。**

In [ ]:
K=5
EPOCHS=4  # 授業時間に合わせて変更可（GPU推奨）
labels_train=np.array([idx_by_name[r["label"]] for r in train_rows])
skf=StratifiedKFold(n_splits=K,shuffle=True,random_state=SEED)
fold_scores=[]
for fold,(tr_idx,val_idx) in enumerate(skf.split(np.zeros(len(train_rows)),labels_train),1):
    torch.manual_seed(SEED+fold)
    model_cv=make_model()
    optimizer=torch.optim.Adam(model_cv.parameters(),lr=1e-3)
    dl_train=loader(Subset(train_aug,tr_idx.tolist()),shuffle=True)
    dl_val=loader(Subset(train_eval,val_idx.tolist()))
    for ep in range(EPOCHS):
        tl,ta=run_epoch(model_cv,dl_train,optimizer)
        vl,va=run_epoch(model_cv,dl_val)
        print(f"Fold {fold}/{K}  epoch {ep+1}/{EPOCHS}  train={ta:.3f}  val={va:.3f}")
    fold_scores.append(va)
    del model_cv,optimizer
print(f"交差検証の平均正解率: {np.mean(fold_scores):.3f} ± {np.std(fold_scores):.3f}")
print("これはtrain内の検証結果です。testの正解率ではありません。")

## 5. 最終CNNを作る
同じ設定で**train全体**を学習し直します。交差検証で使ったモデルはtest評価には流用しません。

In [ ]:
torch.manual_seed(SEED+100)
model=make_model()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
for ep in range(EPOCHS):
    loss,acc=run_epoch(model,loader(train_aug,shuffle=True),optimizer)
    print(f"最終学習 {ep+1}/{EPOCHS} train正解率={acc:.3f} 損失={loss:.4f}")
model.eval()
out_model=Path("/content/tomato_cnn_model.pt")
torch.save({"state_dict":model.state_dict(),"classes":classes,"image_size":SIZE,"epochs":EPOCHS},out_model)
print("学習済みモデル:",out_model)

## 6. 固定testデータで評価
交差検証と最終学習に使っていないtestだけで性能を評価します。

In [ ]:
y_true=[];y_pred=[]
model.eval()
with torch.no_grad():
    for x,y in loader(test_data):
        pred=model(x.to(DEVICE)).argmax(1).cpu().tolist()
        y_true.extend(y.tolist());y_pred.extend(pred)
print("test正解率:",f"{accuracy_score(y_true,y_pred):.3f}")
print(classification_report(y_true,y_pred,labels=list(range(len(classes))),target_names=[JP.get(c,c) for c in classes],zero_division=0))
print("混同行列（行=正解、列=予測）")
print(confusion_matrix(y_true,y_pred))

## 7. 画像判別UI
- 「testからランダムに判別する」：**未学習のtest画像をランダム表示**して正解・予測を比較します。
- 「画像をアップロードして判別する」：自分のJPEG/PNG画像を判別します。アップロード画像には正解ラベルはありません。

※ 病気の判定は学習データに基づく予測です。実際の植物病害の確定診断には利用できません。

In [ ]:
from library_hiroba import ui
from IPython.display import display
import html as htmlmod
def predict_image(pil):
    x=image_to_tensor(pil).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        prob=torch.softmax(model(x),dim=1)[0].cpu().numpy()
    return int(prob.argmax()),prob
def result_ui(pil, true_label=None, source="画像"):
    pred,prob=predict_image(pil)
    preview=pil.copy().convert("RGB"); preview.thumbnail((420,350))
    buf=io.BytesIO();preview.save(buf,format="PNG")
    src=base64.b64encode(buf.getvalue()).decode("ascii")
    name=JP.get(classes[pred],classes[pred])
    correct_html="" if true_label is None else f'<div>正解ラベル：<strong>{htmlmod.escape(JP.get(true_label,true_label))}</strong></div>'
    status="" if true_label is None else ("✅ 正解" if classes[pred]==true_label else "❌ 不正解")
    bars="".join(f'<div style="font-size:13px;margin:4px 0">{htmlmod.escape(JP.get(c,c))}: {float(prob[i])*100:.1f}%</div>' for i,c in enumerate(classes))
    return ui.html(f'<div style="padding:16px;border:1px solid #ccd;border-radius:12px;max-width:560px;background:white;color:#222">'
       f'<div style="font-weight:bold;font-size:18px">{htmlmod.escape(source)} {status}</div>'
       f'<img src="data:image/png;base64,{src}" alt="判別対象の葉" style="display:block;max-width:100%;max-height:350px;object-fit:contain;margin:12px auto">'
       f'{correct_html}<div>予測ラベル：<strong>{htmlmod.escape(name)}</strong></div>'
       f'<div>予測スコア：{float(prob[pred])*100:.1f}%</div><hr>{bars}</div>')
def random_test(_action):
    r=random.choice(test_rows)
    with Image.open(root/r["image"]) as im:
        return result_ui(im.copy(),true_label=r["label"],source="testデータ")
display(ui.card("トマトの葉を判別してみよう","下のボタンでランダムなテスト画像を選び、AIの予測と正解を比べましょう。"))
display(ui.form(random_test,ui.field("_action",label="操作",kind="choice",choices=["ランダムに1枚選ぶ"]),submit_label="testからランダムに判別する",title="テスト画像で試す",pending=None))

In [ ]:
# ファイルのアップロードボタン。結果表示にはLibrary Hirobaのui.htmlを使用します。
import ipywidgets as widgets
from IPython.display import display
upload=widgets.FileUpload(accept="image/png,image/jpeg",multiple=False,description="画像をアップロードして判別する")
upload_result=widgets.Output()
def uploaded(change):
    if not upload.value: return
    try:
        item=next(iter(upload.value.values())) if isinstance(upload.value,dict) else upload.value[0]
        content=item["content"] if isinstance(item,dict) else item.content
        with Image.open(io.BytesIO(bytes(content))) as im:
            widget=result_ui(im.copy(),source="アップロード画像")
        with upload_result:
            upload_result.clear_output(wait=True)
            display(widget)
    except Exception as e:
        with upload_result:
            upload_result.clear_output(wait=True)
            display(ui.alert(f"画像を読み込めませんでした: {e}",kind="danger"))
upload.observe(uploaded,names="value")
display(ui.card("自分の画像を試す","PNGまたはJPEGをアップロードすると、CNNが予測します。"))
display(upload,upload_result)

## 教材としての注意
学習と検証はtrainだけを使用し、最後のtest評価までtestは学習に使用しません。GitHubのmanifestに記載されている重複除去はSHA-256による**完全一致**のみで、同一の葉を異なる角度で撮った画像などの混入を排除できたことは意味しません。

**ライセンス**：このNotebook自体のサンプルコードとは別に、画像データの再利用・再配布は出典および権利関係を確認してください。GitHubのケース別READMEにはライセンスの確定的な表記がありません。